# CNN-Transformer HOSE Training on Colab

This notebook clones the repo, loads the pre-sharded training data prepared for GitHub, trains the CNN-Transformer on GPU if available, evaluates the best checkpoint with batched test inference, exports weekly top-5 stock picks, and plots test-period portfolio performance versus VNINDEX.

In [ ]:
!nvidia-smi || true

In [ ]:
!git clone https://github.com/votranhuonggiang/trading__weekly_CNN_Transformer.git
%cd trading__weekly_CNN_Transformer
!pip install -q -r requirements.txt

In [ ]:
from src.colab_data import load_sharded_arrays
from src.train import TrainingConfig, train_model
from src.model import CNNTransformerClassifier
from src.predict import build_portfolio_performance, build_top_k_portfolio, build_weekly_prediction_table, export_prediction_outputs
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

arrays = load_sharded_arrays('data/colab')
metadata = pd.read_parquet('data/colab/model_dataset_metadata.parquet')
vnindex_comparison = pd.read_csv('outputs/tables/vnindex_label_comparison.csv')

expected_metadata_cols = {
    'rebalance_date',
    'next_rebalance_date',
    'ticker',
    'label',
    'label_name',
    'next_week_return',
    'return_rank_pct',
    'split',
}
legacy_cols = {'triple_barrier_event', 'triple_barrier_return'}
if not expected_metadata_cols.issubset(metadata.columns) or legacy_cols.intersection(metadata.columns):
    raise RuntimeError(
        'Stale data/colab package detected. Run python scripts/prepare_colab_data.py locally, '
        'commit the refreshed data/colab files, then re-clone in Colab. '
        f'Current metadata columns: {metadata.columns.tolist()}'
    )

metadata['rebalance_date'] = pd.to_datetime(metadata['rebalance_date']).dt.normalize()
metadata['next_rebalance_date'] = pd.to_datetime(metadata['next_rebalance_date']).dt.normalize()
vnindex_comparison['rebalance_date'] = pd.to_datetime(vnindex_comparison['rebalance_date']).dt.normalize()
vnindex_comparison['next_rebalance_date'] = pd.to_datetime(vnindex_comparison['next_rebalance_date']).dt.normalize()

print({k: v.shape for k, v in arrays.items() if hasattr(v, 'shape')})
print('metadata_rows:', len(metadata))
print('metadata_cols:', metadata.columns.tolist())
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print('gpu_name:', torch.cuda.get_device_name(0))


In [ ]:
config = TrainingConfig(
    epochs=15,
    batch_size=512 if torch.cuda.is_available() else 128,
    learning_rate=1e-3,
    patience=4,
    checkpoint_dir='checkpoints'
)
model, metrics = train_model(arrays, config)
print('best_epoch:', metrics['best_epoch'])
print('accuracy:', metrics['accuracy'])
print('macro_f1:', metrics['macro_f1'])
print(metrics['report'])
print('checkpoint:', metrics['checkpoint_path'])

In [ ]:
history = metrics['history']
epochs = [row['epoch'] for row in history]
train_loss = [row['train_loss'] for row in history]
val_accuracy = [row['val_accuracy'] for row in history]
val_macro_f1 = [row['val_macro_f1'] for row in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs, train_loss, marker='o', color='#1f4e79')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

axes[1].plot(epochs, val_accuracy, marker='o', label='Val Accuracy', color='#117a65')
axes[1].plot(epochs, val_macro_f1, marker='o', label='Val Macro F1', color='#b03a2e')
axes[1].set_title('Validation Metrics')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
device = torch.device(config.device)
num_classes = int(arrays['y_train'].max()) + 1
best_model = CNNTransformerClassifier(num_features=arrays['X_train'].shape[-1], num_classes=num_classes).to(device)
best_model.load_state_dict(torch.load(metrics['checkpoint_path'], map_location=device))
best_model.eval()

test_batch_size = 1024 if torch.cuda.is_available() else 256
test_loader = DataLoader(
    TensorDataset(
        torch.tensor(arrays['X_test'], dtype=torch.float32),
        torch.tensor(arrays['y_test'], dtype=torch.long),
    ),
    batch_size=test_batch_size,
    shuffle=False,
    num_workers=0,
)

preds = []
probs_list = []
targets = []
with torch.no_grad():
    for x_batch, y_batch in test_loader:
        logits = best_model(x_batch.to(device))
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        pred = np.argmax(probs, axis=1)
        probs_list.append(probs)
        preds.append(pred)
        targets.append(y_batch.numpy())

probs = np.concatenate(probs_list)
preds = np.concatenate(preds)
y_test = np.concatenate(targets)

print('test_accuracy:', accuracy_score(y_test, preds))
print('test_macro_f1:', f1_score(y_test, preds, average='macro'))
print('predicted_class_counts:', dict(zip(['NotBuy', 'Buy'], np.bincount(preds, minlength=2).tolist())))
print(classification_report(y_test, preds, target_names=['NotBuy', 'Buy'], digits=4, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['NotBuy', 'Buy'])
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Test Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
avg_probs = probs.mean(axis=0)
plt.figure(figsize=(7, 4))
plt.bar(['NotBuy', 'Buy'], avg_probs, color=['#7f8c8d', '#117a65'])
plt.title('Average Predicted Probability on Test Set')
plt.ylabel('Probability')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
all_x = np.concatenate([arrays['X_train'], arrays['X_val'], arrays['X_test']], axis=0)
all_probs = []
all_loader = DataLoader(
    TensorDataset(torch.tensor(all_x, dtype=torch.float32)),
    batch_size=1024 if torch.cuda.is_available() else 256,
    shuffle=False,
    num_workers=0,
)

with torch.no_grad():
    for (x_batch,) in all_loader:
        logits = best_model(x_batch.to(device))
        all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())

all_probs = np.concatenate(all_probs)
prediction_table = build_weekly_prediction_table(metadata, all_probs)
top5_table = build_top_k_portfolio(prediction_table, top_k=5)
top5_table['rebalance_date'] = pd.to_datetime(top5_table['rebalance_date']).dt.normalize()
score_path, top5_path = export_prediction_outputs(prediction_table, 'outputs/predictions', top_k=5)
fee_rate = 0.002
portfolio_perf = build_portfolio_performance(top5_table, vnindex_comparison=vnindex_comparison, fee_rate=fee_rate)
portfolio_perf['rebalance_date'] = pd.to_datetime(portfolio_perf['rebalance_date']).dt.normalize()
portfolio_perf.to_csv('outputs/predictions/weekly_top_5_performance.csv', index=False)

test_perf = portfolio_perf[portfolio_perf['rebalance_date'] >= pd.Timestamp('2024-01-01')].copy()
test_perf['portfolio_cumulative_test'] = (1.0 + test_perf['portfolio_simple_return']).cumprod()
if 'vnindex_simple_return' in test_perf.columns and test_perf['vnindex_simple_return'].notna().any():
    test_perf['vnindex_cumulative_test'] = (1.0 + test_perf['vnindex_simple_return']).cumprod()

portfolio_sharpe = np.sqrt(52) * test_perf['portfolio_simple_return'].mean() / test_perf['portfolio_simple_return'].std() if test_perf['portfolio_simple_return'].std() > 0 else np.nan
vnindex_sharpe = np.nan
if 'vnindex_simple_return' in test_perf.columns and test_perf['vnindex_simple_return'].notna().any() and test_perf['vnindex_simple_return'].std() > 0:
    vnindex_sharpe = np.sqrt(52) * test_perf['vnindex_simple_return'].mean() / test_perf['vnindex_simple_return'].std()

print('score_csv:', score_path)
print('top5_csv:', top5_path)
print('performance_csv: outputs/predictions/weekly_top_5_performance.csv')
print('fee_rate:', fee_rate)
print('portfolio_sharpe_test:', portfolio_sharpe)
print('vnindex_sharpe_test:', vnindex_sharpe)
display(test_perf.head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))
ax.plot(test_perf['rebalance_date'], test_perf['portfolio_cumulative_test'], label='Top 5 Portfolio (Test)', color='#117a65', linewidth=2.2)
if 'vnindex_cumulative_test' in test_perf.columns and test_perf['vnindex_cumulative_test'].notna().any():
    ax.plot(test_perf['rebalance_date'], test_perf['vnindex_cumulative_test'], label='VNINDEX (Test)', color='#1f4e79', linewidth=2.0)

ax.set_title('Top 5 Weekly Portfolio vs VNINDEX - Test Period Only')
ax.set_xlabel('Rebalance Date')
ax.set_ylabel('Cumulative Growth of 1.0')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()